# A3.2 · Sandboxed execution

**Function A — Security Architecture & Platform → Controls — Runtime and the Gateway**  ·  *Security of AI*

Builds on **[A3.1 · Default-deny on the tool call](https://spbreed.github.io/cyber-commons/lessons/A3.1.html)**.

| | |
|---|---|
| Open-source tooling | gVisor, Falco |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


**Mitigates: T11 Unexpected Code Execution · T2 Tool Misuse.**

For an agent that runs code, the sandbox **is** the security boundary. Not the
prompt, not the code review, not the model's training. The question is never
"is there a sandbox" but "what does this one actually contain".

A1.8's lesson was that reach is a property of the environment, not of intent —
the benign task touched a private key because `open()` sees what the process
sees. So the control is to change what the process sees.

Four dimensions, and the fourth is the one teams get wrong:

**Filesystem.** A bounded working directory. Not the home directory, which holds
`.ssh`, `.aws` and `.config`.

**Process and syscall.** No spawning, no ptrace, resource ceilings so a runaway
loop is contained rather than fatal.

**Network.** No egress by default. Combined with A3.3 this is what makes an
exfiltration primitive useless.

**Credentials.** The one people miss: **a sandbox with production credentials
mounted in it is not a sandbox.** Isolation of the filesystem is irrelevant if
the environment holds a token that reaches production over a network the
sandbox does permit. The strongest boundary in the world does not help when the
keys are inside it.

> **What this control closes.**
>
> Changes what the executing process can **reach**, which is the only variable A1.8 turned on. A sandbox holding production credentials contains nothing that matters.\n\n## 2 · The control

In [ ]:
HOST = {"fs": ["/home/agent/work/data.csv", "/home/agent/.ssh/id_ed25519",
                "/etc/passwd"],
        "env": {"AWS_ACCESS_KEY_ID": "AKIA-EXAMPLE-NOT-REAL",
                "DATABASE_URL": "postgres://prod-db/main"},
        "net": ["prod-db:5432", "169.254.169.254:80", "0.0.0.0/0"]}

def sandbox(workdir="/sandbox/work", allow_net=(), keep_env=()):
    return {"fs": [p for p in HOST["fs"] if p.startswith(workdir)] + [f"{workdir}/data.csv"],
            "env": {k: v for k, v in HOST["env"].items() if k in keep_env},
            "net": list(allow_net)}

def reach(env, code):
    out = []
    if "open("    in code: out += [f"file:{p}" for p in sorted(env["fs"])]
    if "environ"  in code: out += [f"env:{k}"  for k in sorted(env["env"])]
    if "connect"  in code: out += [f"net:{h}"  for h in sorted(env["net"])]
    return out

CODE = "import os; d=os.environ; open('/home/agent/.ssh/id_ed25519'); connect('x')"

configs = {
 "no sandbox":                    HOST,
 "sandbox, prod creds mounted":   sandbox(allow_net=["prod-db:5432"],
                                          keep_env=("AWS_ACCESS_KEY_ID", "DATABASE_URL")),
 "sandbox, no ambient creds":     sandbox(),
}
for label, env in configs.items():
    r = reach(env, CODE)
    print(f"{label:32s}reached {len(r)}")
    for item in r:
        print(f"      {item}")
    print()

print("The middle configuration is the one that ships. The filesystem is")
print("isolated, the syscalls are filtered, and the environment holds a")
print("credential that reaches production over a network hop the sandbox allows.")
print()
print("Isolation of the wrong dimension is not a weaker control. It is the")
print("appearance of one.")
assert reach(configs["sandbox, no ambient creds"], CODE) and \
       not any("env:" in r for r in reach(configs["sandbox, no ambient creds"], CODE))

## What you just proved

The same code is executed against three environments. Unsandboxed it reaches a private key, two credentials and the whole network. Sandboxed but with production credentials mounted it still reaches both credentials and the production database. Only the third — no ambient credentials — contains it.

## Your turn

Print the environment of one sandbox you run code in. Every credential in it is reachable by anything that executes there, and the isolation you paid for does not apply to any of them.

---

**Next → [A3.3 · Egress control](https://spbreed.github.io/cyber-commons/lessons/A3.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*